In [1]:
# 读取PepSet-bound目录下的子目录，将每个子目录中的pdb文件合并，保存为一个新的pdb文件
import os
import shutil
class Atom():
    '''Process every atom in a PDB file'''
    
    def __init__(self,line):
            self.property = line[0:6].replace(' ','')
            self.atom_num = int(line[6:11])
            self.atom_name = line[12:16].replace(' ','')
            self.alternate_location_indicator = line[16:17]
            self.residue_type = line[17:20].replace(' ','')
            self.chain = line[21:22].replace(' ','')
            self.residue_num = int(line[22:26])
            self.insertion = line[26:27]
            self.x = float(line[30:38])
            self.y = float(line[38:46])
            self.z = float(line[46:54])
            self.occup = float(line[54:60])
            self.tf = float(line[60:66])
            self.segment = line[72:76].replace(' ','')
            self.element = line[76:78]
            self.charge = line[78:80].strip()

    def return_line(self):
        return f'{self.property:<6}{self.atom_num:>5d} {self.atom_name:^4s}{self.alternate_location_indicator}{self.residue_type:3} {self.chain}{self.residue_num:>4}{self.insertion}   {self.x:>8.3f}{self.y:>8.3f}{self.z:>8.3f}{self.occup:>6.2f}{self.tf:>6.2f}      {self.segment:>4s}{self.element:>2s}{self.charge:2s}'

def merge_pdb_files(input_dir, output_dir):
    for subdir in os.listdir(input_dir):
        subdir_path = os.path.join(input_dir, subdir)
        # 读取subdir中的文件，将pep.pdb的内容放在rec_b.pdb文件的后面，新文件名称为当前目录的名称，要求pep.pdb的原子序号延续rec_b.pdb的原子序号，并且删除所有的氢原子，水分子
        if os.path.isdir(subdir_path):
            rec_file = os.path.join(subdir_path, 'rec_b.pdb')
            pep_file = os.path.join(subdir_path, 'pep.pdb')
            output_file = os.path.join(output_dir, f'{subdir}.pdb')
            
            with open(rec_file, 'r') as rf, open(pep_file, 'r') as pf, open(output_file, 'w') as of:
                atom_count = 0
                # 处理受体文件
                for line in rf:
                    if line.startswith('ATOM') or line.startswith('HETATM'):
                        atom = Atom(line)
                        if atom.element != 'H' and atom.residue_type != 'HOH' and atom.atom_name != 'OXT':
                            atom_count += 1
                            atom.atom_num = atom_count
                            of.write(atom.return_line() + '\n')
                # 处理配体文件
                for line in pf:
                    if line.startswith('ATOM') or line.startswith('HETATM'):
                        atom = Atom(line)
                        if atom.element != 'H' and atom.residue_type != 'HOH' and atom.atom_name != 'OXT':
                            atom_count += 1
                            atom.atom_num = atom_count
                            of.write(atom.return_line() + '\n')
                of.write('END\n')


def merge_pdb_files_2(input_dir, output_dir):
    # 读取input_dir下的pdb文件，所有文件命名为PDB_pep或PDB_pro，将相同PDB的pep与pro合并，pep放在pro后，保存为新的pdb文件，文件名为PDB.pdb，要求pep的原子序号延续pro的原子序号，并且删除所有的氢原子，水分子
    # 对于带有带有插入码的残基，保留其中的A插入码的原子，删除其他插入码的原子
    pdb_dict = {}
    for filename in os.listdir(input_dir):
        if filename.endswith('.pdb'):
            pdb_id = filename.split('_')[0]
            if pdb_id not in pdb_dict:
                pdb_dict[pdb_id] = {}
            if 'pep' in filename:
                pdb_dict[pdb_id]['pep'] = os.path.join(input_dir, filename)
            elif 'pro' in filename:
                pdb_dict[pdb_id]['pro'] = os.path.join(input_dir, filename)
    for pdb_id, files in pdb_dict.items():
        if 'pep' in files and 'pro' in files:
            pro_file = files['pro']
            pep_file = files['pep']
            output_file = os.path.join(output_dir, f'{pdb_id}.pdb')
            with open(pro_file, 'r') as pf, open(pep_file, 'r') as ef, open(output_file, 'w') as of:
                atom_count = 0
                # 处理蛋白文件
                for line in pf:
                    if line.startswith('ATOM') or line.startswith('HETATM'):
                        atom = Atom(line)
                        if atom.alternate_location_indicator == ' ' or atom.alternate_location_indicator == 'A': 
                            if atom.element != 'H' and atom.residue_type != 'HOH' and atom.atom_name != 'OXT':
                                atom_count += 1
                                atom.atom_num = atom_count 
                                of.write(atom.return_line())
                # 处理肽文件
                for line in ef:
                    if line.startswith('ATOM') or line.startswith('HETATM'):
                        atom = Atom(line)
                        if atom.alternate_location_indicator == ' ' or atom.alternate_location_indicator == 'A': 
                            if atom.element != 'H' and atom.residue_type != 'HOH' and atom.atom_name != 'OXT':
                                atom_count += 1
                                atom.atom_num = atom_count
                                atom.chain = 'L'
                                of.write(atom.return_line())
                of.write('END\n')

In [16]:
input_directory = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_weight/datasets/PepSet/bound'
output_directory = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_weight/datasets/PepSet/Merge_PDBs'
if not os.path.exists(output_directory):
    os.makedirs(output_directory)
merge_pdb_files(input_directory, output_directory)


In [35]:
input_directory_2 = './datasets/PeptiDB-Tsaban/PeptiDB_Tsaban'
output_directory_2 = './datasets/PeptiDB-Tsaban/PeptiDB_Tsaban_merged'
if not os.path.exists(output_directory_2):
    os.makedirs(output_directory_2)
merge_pdb_files_2(input_directory_2, output_directory_2)


In [5]:
# 对Merged_PDBs目录下的每个pdb文件，进行以下处理：将所有的非L链的残基类型改为UNK，属性改为HETATM，保存为新的pdb文件在Processed_PDBs目录下
def process_pdb_files(input_dir, output_dir):
    for pdb_file in os.listdir(input_dir):
        if pdb_file.endswith('.pdb'):
            input_pdb_path = os.path.join(input_dir, pdb_file)
            output_pdb_path = os.path.join(output_dir, pdb_file)
            
            with open(input_pdb_path, 'r') as infile, open(output_pdb_path, 'w') as outfile:
                for line in infile:
                    if line.startswith('ATOM') or line.startswith('HETATM'):
                        atom = Atom(line)
                        if atom.chain != 'B':
                            atom.residue_type = 'UNK'
                            atom.property = 'HETATM'
                        outfile.write(atom.return_line() + "\n")
                    else:
                        outfile.write(line + "\n")
            print(f'Processed {input_pdb_path} and saved to {output_pdb_path}')

In [6]:
input_directory = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass'
output_directory = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed'
if not os.path.exists(output_directory):
    os.makedirs(output_directory)
process_pdb_files(input_directory, output_directory)

Processed /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass/3obq.pdb and saved to /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed/3obq.pdb
Processed /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass/6fbk.pdb and saved to /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed/6fbk.pdb
Processed /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass/2hpl.pdb and saved to /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed/2hpl.pdb
Processed /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass/2ke1.pdb and saved to /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed/2ke1.

# PDB清洗

In [36]:
import os
# 检查pdb中所有残基是否只包含一个N、CA、C、O，如果不是，则指出文件名和缺失的残基号
def check_pdb_files(input_dir):
    check = True
    for pdb_file in os.listdir(input_dir):
        if pdb_file.endswith('.pdb'):
            residues = {}
            pdb_path = os.path.join(input_dir, pdb_file)
            with open(pdb_path, 'r') as f:
                for line in f:
                    if line.startswith('ATOM') or line.startswith('HETATM'):
                        atom = Atom(line)
                        key = (atom.chain, atom.residue_num)
                        if key not in residues:
                            residues[key] = set()
                        residues[key].add(atom.atom_name)
            for (chain, res_num), atom_names in residues.items():
                required_atoms = {'N', 'CA', 'C', 'O'}
                if atom_names & required_atoms != required_atoms:
                    missing = required_atoms - atom_names
                    print(f"{pdb_file}: Residue {res_num} in chain {chain} missing atoms: {', '.join(missing)}")
                    check = False
        if check:
            print(f"All residues in {pdb_file} have N, CA, C, O atoms.")
                
check_pdb_files('./datasets/PeptiDB-Tsaban/PeptiDB_Tsaban_merged')



All residues in 5njx.pdb have N, CA, C, O atoms.
All residues in 4tzm.pdb have N, CA, C, O atoms.
All residues in 4dcb.pdb have N, CA, C, O atoms.
All residues in 2hpl.pdb have N, CA, C, O atoms.
All residues in 1yuc.pdb have N, CA, C, O atoms.
All residues in 6fkp.pdb have N, CA, C, O atoms.
All residues in 1uj0.pdb have N, CA, C, O atoms.
All residues in 2d0n.pdb have N, CA, C, O atoms.
All residues in 2zjd.pdb have N, CA, C, O atoms.
All residues in 1tw6.pdb have N, CA, C, O atoms.
All residues in 3d9t.pdb have N, CA, C, O atoms.
All residues in 2ak5.pdb have N, CA, C, O atoms.
All residues in 1oai.pdb have N, CA, C, O atoms.
1cka.pdb: Residue 190 in chain A missing atoms: O
1cka.pdb: Residue 9 in chain L missing atoms: O
3brh.pdb: Residue 196 in chain A missing atoms: C, O
4aph.pdb: Residue 617 in chain A missing atoms: O
4aph.pdb: Residue 7 in chain L missing atoms: CA, C, O
6hgt.pdb: Residue 12 in chain L missing atoms: O
6pj8.pdb: Residue 496 in chain L missing atoms: N
1er8.pdb

In [37]:
# 检查pdb中的骨架原子（N、CA、C、O）个数是否为残基个数的四倍，如果不是，则输出该pdb的名称
def check_backbone_atoms(input_dir):
    for pdb_file in os.listdir(input_dir):
        if pdb_file.endswith('.pdb'):
            backbone_count = 0
            residue_set = set()
            pdb_path = os.path.join(input_dir, pdb_file)
            with open(pdb_path, 'r') as f:
                for line in f:
                    if line.startswith('ATOM') or line.startswith('HETATM'):
                        atom = Atom(line)
                        if atom.atom_name in {'N', 'CA', 'C', 'O'}:
                            backbone_count += 1
                            residue_set.add((atom.chain, atom.residue_num))
            residue_count = len(residue_set)
            if backbone_count != residue_count * 4:
                print(f"{pdb_file} has {backbone_count} backbone atoms but {residue_count} residues.")


check_backbone_atoms('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PeptiDB-Tsaban/PeptiDB_Tsaban_merged')

In [15]:
import os
from pathlib import Path

def get_subdirectories(path):
    """
    获取指定路径下的所有第一层子目录名称。
    """
    p = Path(path)
    if not p.exists():
        print(f"错误: 路径不存在 -> {path}")
        return set()
    
    # 遍历路径，筛选出是目录(is_dir)的项目，并获取其名称(name)
    # 使用 set (集合) 存储，方便后续进行数学运算
    return {item.name for item in p.iterdir() if item.is_dir()}

def compare_directories(path_a, path_b):
    print(f"正在比较目录:\n A: {path_a}\n B: {path_b}\n" + "-"*30)

    # 1. 获取两个目录下的文件夹名称集合
    dirs_a = get_subdirectories(path_a)
    dirs_b = get_subdirectories(path_b)

    if not dirs_a and not dirs_b:
        print("警告: 两个目录似乎都是空的或不存在，无法比较。")
        return

    # 2. 计算差异 (集合差集运算)
    # 在 A 中存在，但在 B 中缺失的
    missing_in_b = dirs_a - dirs_b
    
    # 在 B 中存在，但在 A 中缺失的
    missing_in_a = dirs_b - dirs_a

    # 3. 输出结果
    print(f"### 比较结果 ###")
    
    if missing_in_b:
        print(f"\n[目录 B] 中缺失 (但在 A 中存在) 的目录有 {len(missing_in_b)} 个:")
        for name in sorted(missing_in_b):
            print(f"  - {name}")
    else:
        print("\n目录 B 没有缺失目录 (A 中的所有目录 B 里都有)。")

    if missing_in_a:
        print(f"\n[目录 A] 中缺失 (但在 B 中存在) 的目录有 {len(missing_in_a)} 个:")
        for name in sorted(missing_in_a):
            print(f"  - {name}")
    else:
        print("\n目录 A 没有缺失目录 (B 中的所有目录 A 里都有)。")

# --- 使用示例 ---
# 请在这里替换为你实际想要比较的两个路径
# Windows 示例: r"C:\Projects\Backup_v1"
# Mac/Linux 示例: "/home/user/projects/v1"

directory_1 = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/PepSet-bound" 
directory_2 = "/home/junjiechen/2_share_database/pepset/bound"

# 运行比较函数
if __name__ == "__main__":
    # 为了演示，如果不修改路径可能会报错，这里加个简单的检查
    compare_directories(directory_1, directory_2)

正在比较目录:
 A: /home/junjiechen/1_work/250401-Dpepalign/Benchmark/PepSet-bound
 B: /home/junjiechen/2_share_database/pepset/bound
------------------------------
### 比较结果 ###

目录 B 没有缺失目录 (A 中的所有目录 B 里都有)。

[目录 A] 中缺失 (但在 B 中存在) 的目录有 4 个:
  - 2hwn
  - 4cc2
  - 5ajn
  - 6n3e


In [7]:
# 检查pdb包含的链个数，分类将不同链数的pdb文件名保存在不同的list中
import os
def classify_pdb_by_chain_count(input_dir):
    chain_count_dict = {}
    for pdb_file in os.listdir(input_dir):
        if pdb_file.endswith('.pdb'):
            chains = set()
            pdb_path = os.path.join(input_dir, pdb_file)
            with open(pdb_path, 'r') as f:
                for line in f:
                    if line.startswith('ATOM') or line.startswith('HETATM'):
                        atom = Atom(line)
                        chains.add(atom.chain)
            chain_count = len(chains)
            if chain_count not in chain_count_dict:
                chain_count_dict[chain_count] = []
            chain_count_dict[chain_count].append(pdb_file)
    return chain_count_dict
chain_classification = classify_pdb_by_chain_count('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PeptiDB-Tsaban/PeptiDB_Tsaban')
for key, value in chain_classification.items():
    if key != 1:
        for i in value:
            print(i)

